<!-- # FusionMamba: Efficient Remote Sensing Image Fusion with State Space Model -->
# FusionMamba：高效遥感图像融合与状态空间模型

- [论文](https://arxiv.org/abs/2404.07932)
- [Github代码](https://github.com/PSRben/FusionMamba)

## 3 方法论

### 3.2 预备赛

#### 3.2.1 状态空间模型(SSM)

**状态空间模型（SSM）** 为连续系统，其通过 $N$ 维中间隐藏状态 $h(t) \in \mathbb{R}^N$，实现一维输入 $x(t) \in \mathbb{R}$ 到一维输出 $y(t) \in \mathbb{R}$ 的映射。该过程通常由如下常微分方程（ODE）描述：

$$
\begin{aligned}
    h'(t)&=\mathbf{A}h(t)+\mathbf{B}x(t), \\
    y(t)&=\mathbf{C}h(t).
\end{aligned} \tag{1}
$$

其中，状态矩阵 $\mathbf{A} \in \mathbb{R}^{N \times N}$ 主导系统演化；投影参数 $\mathbf{B} \in \mathbb{R}^{N \times 1}$ 与 $\mathbf{C} \in \mathbb{R}^{1 \times N}$ 调控系统更新。[式1]() 表明，SSM 具备**全局感知能力**——当前输出受所有历史输入的共同影响。
当 $\mathbf{A}$、$\mathbf{B}$、$\mathbf{C}$ 为常数时，该方程表征**线性时不变（LTI）系统**（如[LSSL]()与[S4]()）；若参数随时间动态变化，则对应**线性时变（LTV）系统**（典型代表为[Mamba]()）。
LTI 系统本质上不具备输入内容感知能力，而**输入感知型 LTV 系统**被设计为具备该核心能力。

#### 3.2.2 离散化

在深度学习（DL）领域应用状态空间模型（SSM）时，**离散化操作是必要前提**。为实现该过程，引入时间尺度参数 $\mathbf{\Delta} \in \mathbb{R}$，将连续域参数 $\mathbf{A}$ 和 $\mathbf{B}$ 转换为对应的离散域参数 $\mathbf{\overline{A}}$ 和 $\mathbf{\overline{B}}$。
采用**零阶保持（ZOH）方法**作为变换算法，离散域参数的计算式如下：

$$
\begin{aligned}
    \mathbf{\overline{A}}&=\exp(\mathbf{\Delta}\mathbf{A}), \\
    \mathbf{\overline{B}}&={(\mathbf{\Delta}\mathbf{A})}^{-1}\left(\exp(\mathbf{\Delta}\mathbf{A})-\mathbf{I}\right)\cdot \mathbf{\Delta}\mathbf{B}\approx\mathbf{\Delta}\mathbf{B}.
\end{aligned} \tag{2}
$$

此时，[式1]() 的离散形式可表示为：
$$
\begin{aligned}
    h_t&=\mathbf{\overline{A}}h_{t-1}+\mathbf{\overline{B}}x_{t}, \\
    y_t&=\mathbf{C}h_t.
\end{aligned} \tag{3}
$$

在实际应用中，输入 $x_t$ 为含 $C$ 个分量的特征向量，[式3]() 对各分量执行**独立并行处理**。

#### 3.2.3 选择性扫描

在 Mamba 模型中，**参数随输入动态变化**的特性导致[式3]() 无法被重构为卷积形式，进而阻碍了状态空间模型（SSM）的并行化计算。
为解决这一问题，Mamba 提出了**选择性扫描（selective scan）机制**，该机制整合了三项硬件友好型优化技术：**核融合（kernel fusion）**、**并行扫描（parallel scan）**与**重计算（recomputation）**。
通过选择性扫描，Mamba 得以在保持较低内存占用的同时，实现卓越的计算速度。


## 3.3 网络架构

为充分挖掘状态空间模型（SSM）在遥感图像融合任务中的潜力，本文设计了一种可解释性网络架构。该架构包含四大核心组件：用于特征提取的两个 U 型网络分支（空间分支与光谱分支）、用于信息融合的组合分支，以及用于光谱增强的多尺度上下文注意力（MCA）模块（整体流程如[图3]()所示）。下文将对各网络组件展开详细阐述。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/u2net2.png" />
    <span style="font-size: 12px; color: black;">图3</strong>：拟议的网络架构。我们的设计包括两个 U 形网络分支，用于特征提取，一个组合分支用于信息整合，以及一个 MCA 模块用于谱增强。曼巴和融合曼巴区块的详细结构见图 4。</span>
</div>

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/mamba3.png" />
    <span style="font-size: 12px; color: black;">图4</strong>：双向Mamba块、四向Mamba块和拟议的融合Mamba块的示意图，以及四个压平方向的示意图。FSSM 代表聚空间模型。此外，SSM 和 FSSM 模块的具体细节分别详见算法1 和算法2。</span>
</div>

<!-- **Algorithm 1** SSM Block

**Input**: $\mathbf{x}: \color{green}(HW, C)$  
**Output**: $\mathbf{y}: \color{green}(HW, C)$  

1.  $\mathbf{A}: \color{green}(C, N)$ $\leftarrow \mathbf{Parameter}_{\rm{A}}$  
    *# $\mathbf{A}$ represents $\color{green}C$ sets of structured $\color{green}N\times N$ [matrices]()*
2.  $\mathbf{B}: \color{green}(HW, N)$ $\leftarrow \mathbf{Linear}_{\rm{B}}(\mathbf{x})$  
3.  $\mathbf{C}: \color{green}(HW, N)$ $\leftarrow \mathbf{Linear}_{\rm{C}}(\mathbf{x})$  
4.  $\mathbf{\Delta}: \color{green}(HW, C)$ $\leftarrow {\rm{log}}(1+{\rm{exp}}(\mathbf{Linear}_{\rm{\Delta}}(\mathbf{x})+\mathbf{Parameter}_{\rm{\Delta}}))$  
    *# $\mathbf{Parameter}_{\rm{\Delta}}$ is a bias vector with a size of $\color{green}C$*
5.  $\mathbf{\overline{A}}: \color{green}(HW, C, N)$ $\leftarrow {\rm{exp}}(\mathbf{\Delta}\otimes \mathbf{A})$  
6.  $\mathbf{\overline{B}}: \color{green}(HW, C, N)$ $\leftarrow \mathbf{\Delta}\otimes \mathbf{B}$  
7.  $\mathbf{y} \leftarrow {\rm{SSM}}(\mathbf{\overline{A}}, \mathbf{\overline{B}}, \mathbf{C})(\mathbf{x})$  
    *# $\rm{SSM}$ represents Eq. (3) implemented using selective scan*
8.  **Return** $\mathbf{y}$ -->

**Algorithm 1** SSM Block

**Input**: $\mathbf{x}: \color{green}(HW, C)$  # HW是展平后的空间维度，C是通道数  
**Output**: $\mathbf{y}: \color{green}(HW, C)$  

1.  $\mathbf{A}: \color{green}(C, N)$ $\leftarrow \mathbf{Parameter}_{\rm{A}}$  
    *# $\mathbf{A}$ represents $\color{green}C$ sets of structured $\color{green}N\times N$ [matrices]()*  
    此处采用结构化矩阵（对角矩阵、循环矩阵等）所以是$(CN)$而非$(CN^2)$
2.  $\mathbf{B}: \color{green}(HW, N)$ $\leftarrow \mathbf{Linear}_{\rm{B}}(\mathbf{x})$ 维度变换：$(HW, C) \xrightarrow{\text{Linear}_B} (HW, N)$  
3.  $\mathbf{C}: \color{green}(HW, N)$ $\leftarrow \mathbf{Linear}_{\rm{C}}(\mathbf{x})$ 维度变换：$(HW, C) \xrightarrow{\text{Linear}_C} (HW, N)$  
4.  $\mathbf{\Delta}: \color{green}(HW, C)$ $\leftarrow {\rm{log}}(1+{\rm{exp}}(\mathbf{Linear}_{\rm{\Delta}}(\mathbf{x})+\mathbf{Parameter}_{\rm{\Delta}}))$  
    维度变换：$(HW, C) \xrightarrow{\text{Linear}_{\rm{\Delta}}} (HW, C)$  
    *# $\mathbf{Parameter}_{\rm{\Delta}}$ is a bias vector with a size of $\color{green}C$*  
5.  $\mathbf{\overline{A}}: \color{green}(HW, C, N)$ $\leftarrow {\rm{exp}}(\mathbf{\Delta}\otimes \mathbf{A})$  
    $\otimes$ 哈达玛积，广播机制将 $\mathbf{\Delta}$ 的维度 $(HW, C)$ 扩展为 $(HW, C, N)$  
    将 $\mathbf{A}$ 的维度 $(C, N)$ 扩展为 $(HW, C, N)$，然后逐元素相乘
6.  $\mathbf{\overline{B}}: \color{green}(HW, C, N)$ $\leftarrow \mathbf{\Delta}\otimes \mathbf{B}$  
7.  $\mathbf{y} \leftarrow {\rm{SSM}}(\mathbf{\overline{A}}, \mathbf{\overline{B}}, \mathbf{C})(\mathbf{x})$  
    *# $\rm{SSM}$ represents Eq. (3) implemented using selective scan*
8.  **Return** $\mathbf{y}$

**Algorithm 2** FSSM Block

**Inputs**: $\mathbf{x}^{\rm{a}}, \mathbf{x}^{\rm{b}}: \color{green}(HW, C)$  
**Output**: $\mathbf{y}^{\rm{a}}: \color{green}(HW, C)$  

1.  $\mathbf{A}: \color{green}(C, N) \leftarrow \mathbf{Parameter}_{\rm{A}}$  
    *# $\mathbf{A}$ represents $\color{green}C$ sets of structured $\color{green}N\times N$ [matrices]()*  
    此处采用结构化矩阵（对角矩阵、循环矩阵等）所以是$(CN)$而非$(CN^2)$
2.  $\mathbf{B}: \color{green}(HW, N)$ $\leftarrow \mathbf{Linear}_{\rm{B}}(\mathbf{x}^{\rm{b}})$  
3.  $\mathbf{C}: \color{green}(HW, N)$ $\leftarrow \mathbf{Linear}_{\rm{C}}(\mathbf{x}^{\rm{b}})$  
4.  $\mathbf{\Delta}: \color{green}(HW, C)$ $\leftarrow {\rm{log}}(1+{\rm{exp}}(\mathbf{Linear}_{\rm{\Delta}}(\mathbf{x}^{\rm{b}})+\mathbf{Parameter}_{\rm{\Delta}}))$  
    *# $\mathbf{Parameter}_{\rm{\Delta}}$ is a bias vector with a size of $\color{green}C$*
5.  $\mathbf{\overline{A}}: \color{green}(HW, C, N)$ $\leftarrow {\rm{exp}}(\mathbf{\Delta}\otimes \mathbf{A})$  
6.  $\mathbf{\overline{B}}: \color{green}(HW, C, N)$ $\leftarrow \mathbf{\Delta}\otimes \mathbf{B}$  
7.  $\mathbf{y}^{\rm{a}} \leftarrow {\rm{SSM}}(\mathbf{\overline{A}}, \mathbf{\overline{B}}, \mathbf{C})(\mathbf{x}^{\rm{a}})$  
    *# $\rm{SSM}$ represents Eq. (3) implemented using selective scan*
8.  **Return** $\mathbf{y}^{\rm{a}}$

#### 3.3.1 U型网络分支

该设计支持**分离式、分层高效学习**空间与光谱信息：空间分支专注于从全色图像 $\mathbf{P}$ 中提取空间细节，光谱分支则专门用于从多光谱图像 $\mathbf{M}$ 中捕捉光谱特征。为在不显著增加网络参数量的前提下获取充足的深层信息，我们提取三种不同尺度的特征，这使得每个U型网络分支共包含五个阶段。
在每个阶段中，空间或光谱特征图先经**四向Mamba模块**处理，输出特征图再传入**FusionMamba模块**生成融合结果，随后与原始输入进行残差连接；最后通过不同类型的卷积层，完成空间分辨率与通道数的调整。

#### 3.3.2 组合分支

该设计可实现空间与光谱信息的**全面整合**。为与U型网络分支对齐，组合分支内置五个FusionMamba模块——每个模块接收对应尺度的空间、光谱特征图作为输入，生成融合输出后与原始输入进行残差连接。从全局视角来看，组合分支有效模拟了不同特征的**渐进式融合过程**。

#### 3.3.3 Mamba驱动的通道注意力（MCA）

MCA模块旨在提升光谱信息的表征能力。该模块以主流通道注意力机制为基础，将原机制中的多层感知机（MLP）替换为**双向Mamba模块**，并针对状态空间模型（SSM）的数据处理特性进行了多项优化，具体流程如下：
1.  对多光谱特征图 $\mathbf{M}^{\rm{U}}$ 执行全局最大池化，去除空间信息，得到尺寸为 $1\times 1\times S$ 的特征图；
2.  将其重塑为 $S\times 1$ 的一维序列，通过全连接层将通道数扩充至 $C$；
3.  扩充后的一维序列传入双向Mamba模块提取光谱特征；
4.  输出特征经投影与重塑，还原为 $1\times 1\times S$ 的特征图，最终与组合分支的顶层特征 $\mathbf{F}^{\rm{c}}_5$ 逐通道相乘，完成光谱增强。

### 3.4 Mamba和FusionMamba块

本节详细介绍了双向Mamba块、四向Mamba块和拟议中的FusionMamba块，见[图4]()。此外，我们比较了不同DL模型所需的FLOPs。

#### 3.4.1 双向Mamba块

对于输入一维序列 $\mathbf{x}_{\rm{in}} \in \mathbb{R}^{S \times C}$，首先对其执行层归一化，再通过两个并行全连接层生成两个独立序列 $\mathbf{x} \in \mathbb{R}^{S \times C}$ 与 $\mathbf{z} \in \mathbb{R}^{S \times C}$，数学表达式如下：
$$
\begin{aligned}
    \mathbf{x}, \mathbf{z} = \mathbf{Linear}_{\rm{x}}(\mathbf{Norm}(\mathbf{x}_{\rm{in}})), \mathbf{Linear}_{\rm{z}}(\mathbf{Norm}(\mathbf{x}_{\rm{in}})).
\end{aligned} \tag{4}
$$
其中，$\mathbf{Norm}$ 代表层归一化操作，$\mathbf{Linear}_{\rm{x}}$ 与 $\mathbf{Linear}_{\rm{z}}$ 为两个独立的全连接层。
随后，将序列 $\mathbf{x}$ 沿序列维度翻转得到同维度序列 $\mathbf{\hat{x}}$，并将 $\mathbf{x}$ 与 $\mathbf{\hat{x}}$ 分别输入两个独立的 SSM 模块（详见[算法1]()）进行特征提取，得到输出序列 $\mathbf{\overline{y}}$ 与 $\mathbf{\hat{y}}$，过程如下：
$$
\begin{aligned}
    \mathbf{\hat{x}}&={\rm{VFlip}}(\mathbf{x}), \\
    \mathbf{\overline{y}}, \mathbf{\hat{y}}&={\mathbf{SSM}}_1(\mathbf{x}), {\mathbf{SSM}}_2(\mathbf{\hat{x}}).
\end{aligned} \tag{5}
$$
式中，${\rm{VFlip}}$ 表示序列维度的翻转操作，${\mathbf{SSM}}_1$ 与 ${\mathbf{SSM}}_2$ 为两个独立的 SSM 模块。
接下来，将 $\mathbf{\hat{y}}$ 沿序列维度翻转后与 $\mathbf{\overline{y}}$ 相加，得到序列 $\mathbf{y} \in \mathbb{R}^{S \times C}$。该序列经 $\mathbf{z}$ 门控调制后，输入全连接层进行变换，最终与原始输入 $\mathbf{x}_{\rm{in}}$ 执行残差连接，得到模块输出 $\mathbf{x}_{\rm{out}} \in \mathbb{R}^{S \times C}$，计算式为：
$$
\begin{aligned}
    \mathbf{y}&=\mathbf{\overline{y}} + {\rm{VFlip}}(\mathbf{\hat{y}}), \\
    \mathbf{x}_{\rm{out}}&={\mathbf{Linear}}_{\rm{o}}(\mathbf{y}\cdot {\rm{SiLU}}(\mathbf{z}))+\mathbf{x}_{\rm{in}}.
\end{aligned} \tag{6}
$$
其中，$\mathbf{Linear}_{\rm{o}}$ 为全连接层，${\rm{SiLU}}$ 代表 SiLU 激活函数，$\cdot$ 表示逐元素乘积。


#### 3.4.2 四向Mamba块

对于输入特征图 $\mathbf{F}_{\rm{in}} \in \mathbb{R}^{H \times W \times C}$，先对其执行层归一化，再通过两个并行的 $1\times1$ 卷积层，生成两个独立的特征图 $\mathbf{X} \in \mathbb{R}^{H \times W \times C}$ 与 $\mathbf{Z} \in \mathbb{R}^{H \times W \times C}$，数学表达式如下：
$$
\begin{aligned}
    \mathbf{X}, \mathbf{Z} = \mathbf{Conv}_{\rm{x}}(\mathbf{Norm}(\mathbf{F}_{\rm{in}})), \mathbf{Conv}_{\rm{z}}(\mathbf{Norm}(\mathbf{F}_{\rm{in}})).
\end{aligned} \tag{7}
$$
其中，$\mathbf{Conv}_{\rm{x}}$ 与 $\mathbf{Conv}_{\rm{z}}$ 为两个独立的 $1\times1$ 卷积层，$\mathbf{Norm}$ 代表层归一化操作。

随后，将特征图 $\mathbf{X}$ 沿**四个方向分别展平**，得到四个维度均为 $HW \times C$ 的一维序列 $\mathbf{x}_1, \mathbf{x}_2, \mathbf{x}_3, \mathbf{x}_4$；各序列经独立的 SSM 模块处理后，输出对应序列 $\mathbf{y}_1, \mathbf{y}_2, \mathbf{y}_3, \mathbf{y}_4$，过程如下：
$$
\begin{cases}
    {\mathbf{x}_i}={\rm{Flatten}}_i(\mathbf{X}), \\
    {\mathbf{y}_i}={\mathbf{SSM}}_i(\mathbf{x}_i).
\end{cases} \tag{8}
\quad i = 1, 2, 3, 4
$$
式中，${\rm{Flatten}}_i$ 表示沿第 $i$ 个方向的展平操作，${\mathbf{SSM}}_i$ 为第 $i$ 个独立的 SSM 模块。

接下来，对各 SSM 模块的输出执行**逆展平操作**，并将结果逐元素相加，得到特征图 $\mathbf{Y} \in \mathbb{R}^{H \times W \times C}$。该特征图经 $\mathbf{Z}$ 门控调制后，输入 $1\times1$ 卷积层进行变换，最终与原始输入 $\mathbf{F}_{\rm{in}}$ 执行残差连接，得到模块输出 $\mathbf{F}_{\rm{out}} \in \mathbb{R}^{H \times W \times C}$，计算式为：
$$
\begin{aligned}
    \mathbf{Y}&=\sum_{i=1}^{4}{\rm{Unflatten}}_i(\mathbf{y}_i), \\
    \mathbf{F}_{\rm{out}}&={\mathbf{Conv}}_{\rm{o}}(\mathbf{Y}\cdot {\rm{SiLU}}(\mathbf{Z}))+\mathbf{F}_{\rm{in}}.
\end{aligned} \tag{9}
$$
其中，${\rm{Unflatten}}_i$ 表示沿第 $i$ 个方向的逆展平操作，$\mathbf{Conv}_{\rm{o}}$ 为 $1\times1$ 卷积层，$\cdot$ 为逐元素乘积，${\rm{SiLU}}$ 代表 SiLU 激活函数。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/flops_new.png" />
    <span style="font-size: 12px; color: black;">图5</strong>：卷积层、双向（BD）Mamba 模块、四向（FD）Mamba 模块、FusionMamba 模块以及自/交叉注意力模块在不同空间分辨率下的 FLOP 比较。为了获得最佳视觉效果，我们将D、C、N配置为0.5M、256和64。</span>
</div>

#### 3.4.3 FusionMamba块

原始状态空间模型（SSM）仅支持单输入。为实现不同类型信息的高效融合，我们将其扩展为双输入结构，提出**融合状态空间模型（Fusion State Space Model, FSSM）**，具体实现详见[算法2]()。在FSSM模块中，一个输入负责生成投影参数与时间尺度参数，另一个输入为待处理序列。FusionMamba模块采用对称结构设计，内部包含8个FSSM模块。

对于输入的空间特征图 $\mathbf{F}_{\rm{in}}^{\rm{a}} \in \mathbb{R}^{H \times W \times C}$ 与光谱特征图 $\mathbf{F}_{\rm{in}}^{\rm{b}} \in \mathbb{R}^{H \times W \times C}$，我们借鉴四向 Mamba 模块的设计思路，分别生成两组特征图，计算式如下：
$$
\begin{aligned}
    \mathbf{X}^{\rm{a}}, \mathbf{Z}^{\rm{a}} = \mathbf{Conv}_{\rm{x}}^{\rm{a}}(\mathbf{Norm}(\mathbf{F}_{\rm{in}}^{\rm{a}})), \mathbf{Conv}_{\rm{z}}^{\rm{a}}(\mathbf{Norm}(\mathbf{F}_{\rm{in}}^{\rm{a}})); \\
    \mathbf{X}^{\rm{b}}, \mathbf{Z}^{\rm{b}} = \mathbf{Conv}_{\rm{x}}^{\rm{b}}(\mathbf{Norm}(\mathbf{F}_{\rm{in}}^{\rm{b}})), \mathbf{Conv}_{\rm{z}}^{\rm{b}}(\mathbf{Norm}(\mathbf{F}_{\rm{in}}^{\rm{b}})). \\
\end{aligned} \tag{10}
$$
该式为[式7]()的直接扩展，符号含义不再赘述。

随后，将 $\mathbf{X}^{\rm{a}}$ 与 $\mathbf{X}^{\rm{b}}$ 分别沿四个方向展平，得到的一维序列输入 FSSM 模块完成信息融合，过程如下：
$$
    \begin{cases}
        {\mathbf{x}_i^{\rm{a}}},{\mathbf{x}_i^{\rm{b}}}={\rm{Flatten}}_i(\mathbf{X}^{\rm{a}}), {\rm{Flatten}}_i(\mathbf{X}^{\rm{b}}), \\
        {\mathbf{y}_i^{\rm{a}}},{\mathbf{y}_i^{\rm{b}}}={\mathbf{FSSM}}_i^{\rm{a}}(\mathbf{x}_i^{\rm{a}}, \mathbf{x}_i^{\rm{b}}), {\mathbf{FSSM}}_i^{\rm{b}}(\mathbf{x}_i^{\rm{b}}, \mathbf{x}_i^{\rm{a}}).
    \end{cases}
\quad i = 1, 2, 3, 4 \tag{11}
$$
其中，$\mathbf{FSSM}^{\rm{a}}$ 与 $\mathbf{FSSM}^{\rm{b}}$ 分别对应图[mamba]()中FusionMamba模块的左半部分与右半部分FSSM模块。

接下来，对两组输出分别执行逆展平与求和操作，得到特征图 $\mathbf{Y}^{\rm{a}} \in \mathbb{R}^{H \times W \times C}$ 与 $\mathbf{Y}^{\rm{b}} \in \mathbb{R}^{H \times W \times C}$；两组特征图经门控、卷积与残差连接后，最终通过卷积层融合得到输出 $\mathbf{F}_{\rm{out}}$，计算式为：
$$
    \begin{aligned}
        \mathbf{Y}^{\rm{a}}, \mathbf{Y}^{\rm{b}}&=\sum_{i=1}^{4}{\rm{Unflatten}}_i(\mathbf{y}_i^{\rm{a}}),\sum_{i=1}^{4}{\rm{Unflatten}}_i(\mathbf{y}_i^{\rm{b}}), \\
        \mathbf{F}_{\rm{out}}^{\rm{a}}&={\mathbf{Conv}}_{\rm{o}}^{\rm{a}}(\mathbf{Y}^{\rm{a}}\cdot {\rm{SiLU}}(\mathbf{Z}^{\rm{a}}))+\mathbf{F}_{\rm{in}}^{\rm{a}}, \\
        \mathbf{F}_{\rm{out}}^{\rm{b}}&={\mathbf{Conv}}_{\rm{o}}^{\rm{b}}(\mathbf{Y}^{\rm{b}}\cdot {\rm{SiLU}}(\mathbf{Z}^{\rm{b}}))+\mathbf{F}_{\rm{in}}^{\rm{b}}, \\
        \mathbf{F}_{\rm{out}}&=\mathbf{Conv}_{\rm{o}}(\mathbf{F}_{\rm{out}}^{\rm{a}} + \mathbf{F}_{\rm{out}}^{\rm{b}}).
    \end{aligned} \tag{12}
$$
式中，$\mathbf{Conv}_{\rm{o}}^{\rm{a}}$、$\mathbf{Conv}_{\rm{o}}^{\rm{b}}$ 与 $\mathbf{Conv}_{\rm{o}}$ 均为 $1\times1$ 卷积层，分别用于生成 $\mathbf{F}_{\rm{out}}^{\rm{a}}$、$\mathbf{F}_{\rm{out}}^{\rm{b}}$ 与最终输出 $\mathbf{F}_{\rm{out}}$；$\cdot$ 表示逐元素乘积，$+$ 表示逐元素相加。

#### 3.4.4 FLOPs分析

对于含 $D$ 个参数的卷积层，其计算量通常为 $2HWD$。
已知选择性扫描的计算量为 $9HWCN$，因此参数规模同为 $D$ 的**双向 Mamba 模块**、**四向 Mamba 模块**与**FusionMamba 模块**，其总计算量分别为 $2HWD+18HWCN$、$2HWD+36HWCN$ 与 $2HWD+72HWCN$。
而 Transformer 中的自/交叉注意力模块，总计算量约为 $2HWD+4H^2W^2C$。

各模块的计算量对比结果如[图5]()所示，结果表明：**Mamba 系列模块与 FusionMamba 模块的计算量与卷积层相当，且远低于自/交叉注意力模块**。需特别指出的是，尽管卷积层计算量最低，但其不具备全局信息捕捉能力。

### 3.5 损失函数

本研究的核心贡献在于状态空间模型（SSM）的应用与创新，因此网络训练采用最简单的$\ell_1$ 损失函数，定义如下：
$$
    \begin{aligned}
        \mathcal{Loss} = \frac{1}{T}\sum_{i=1}^{T}{\| f_{\mathbf{\Theta}}(\mathbf{P}_i,\mathbf{M}_i) - \mathbf{O}_i\|}_{1}.
    \end{aligned} \tag{13}
$$
其中，$T$ 为训练样本总数，$f_{\mathbf{\Theta}}$ 代表含可学习参数 $\mathbf{\Theta}$ 的本文所提网络。$\mathbf{P}_i$、$\mathbf{M}_i$、$\mathbf{O}_i$ 分别对应训练集中第 $i$ 个全色（PAN）图像、低分辨率多光谱（LRMS）/低分辨率高光谱（LRHS）图像与真实标签（GT）图像。$\|\cdot\|_1$ 表示 $\ell_1$归一化。

## 4 实验

本节针对**全色锐化**与**高光谱全色锐化**两类典型遥感图像融合任务，给出了主流方法的定量与定性评估结果。同时，通过全面的**消融实验**，验证了本文所提方法的优越性。